## First

In [1]:
import os
from lm_studio_code import qwen
import json
import tqdm
from comfy_code import app
import time
from upload_to_shopify import db, shopify
import shutil
import sys, threading
from PyQt5.QtWidgets import QApplication
from upload_to_shopify import product_editor as pe

# Root directory where this script is located
ROOT_DIR = os.getcwd()

DOWNLOADS_DIR = os.path.join(ROOT_DIR, "downloads", "images")
ORIGINAL_IMAGES_DIR = os.path.join(ROOT_DIR, "original_images")
GENERATED_IMAGES_DIR = os.path.join(ROOT_DIR, "generated_images")
COMFY_OUTPUT_IMAGES_DIR = os.path.join(ROOT_DIR, "ComfyUI_windows_portable", "ComfyUI","output")

print("DOWNLOADS_DIR:", DOWNLOADS_DIR)
print("ORIGINAL_IMAGES_DIR:", ORIGINAL_IMAGES_DIR)
print("GENERATED_IMAGES_DIR:", GENERATED_IMAGES_DIR)
print("COMFY_OUTPUT_IMAGES_DIR:", COMFY_OUTPUT_IMAGES_DIR)

app.start_comfyui()
products_db = db.ProductsDB()
sp = shopify.ShopifyUploader(products_db)

DOWNLOADS_DIR: c:\Users\FIRST\Desktop\piyagushopify\llmautomation\downloads\images
ORIGINAL_IMAGES_DIR: c:\Users\FIRST\Desktop\piyagushopify\llmautomation\original_images
GENERATED_IMAGES_DIR: c:\Users\FIRST\Desktop\piyagushopify\llmautomation\generated_images
COMFY_OUTPUT_IMAGES_DIR: c:\Users\FIRST\Desktop\piyagushopify\llmautomation\ComfyUI_windows_portable\ComfyUI\output
Starting ComfyUI...
COMFY_ROOT: C:\Users\FIRST\Desktop\piyagushopify\llmautomation\ComfyUI_windows_portable
ComfyUI is ready
Authenticated with Shopify at 2026-06-19 01:06:14.643773
Fetched 24 collections from Shopify at 2026-06-19 01:06:15.061233
fetched publications : ['gid://shopify/Publication/171978195083', 'gid://shopify/Publication/171978260619', 'gid://shopify/Publication/172730908811', 'gid://shopify/Publication/175160459403']


In [2]:
ALL_IMAGES = []
for file in os.listdir(DOWNLOADS_DIR):
    if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        ALL_IMAGES.append(os.path.join(DOWNLOADS_DIR, file))
print("No of Images:",len(ALL_IMAGES))

No of Images: 27


In [3]:
qwen.load_model_if_needed()
products = []
for image_path in tqdm.tqdm(ALL_IMAGES):
    response = qwen.generate_response(image_path)
    content = next(
        x["content"]
        for x in response["output"]
        if x["type"] == "message"
    )
    product = json.loads(content)
    product["image_path"] = image_path
    products.append(product)
qwen.unload_model()

Model already loaded: qwen3.5-35b-a3b


100%|██████████| 27/27 [1:14:28<00:00, 165.51s/it]


{'instance_id': 'qwen3.5-35b-a3b'}

In [4]:
workflow = app.load_workflow()
comfy_prompts = app.load_prompts()
for i, product in enumerate(tqdm.tqdm(products)):
    generated_image_prefix = f"generated_{int(time.time() * 1000)}_{i}_"

    prompt = comfy_prompts[product["product_type"]].replace(
        "{set_includes}",
        ", ".join(product["set_includes"])
    )

    app.run_workflow(
        image_path=product["image_path"],
        prompt=prompt,
        workflow=workflow,
        output_prefix=generated_image_prefix
    )

    product["GENERATED_IMAGE_LOCATION"] = os.path.join(COMFY_OUTPUT_IMAGES_DIR,generated_image_prefix+"_00001_.png")
app.free_resources()

  0%|          | 0/27 [00:00<?, ?it/s]

Prompt ID: e963932a-cc1c-477a-bd39-d9d95be370ad


  4%|▎         | 1/27 [02:24<1:02:41, 144.66s/it]

Generation completed
Prompt ID: c9f2c83f-f540-4e3c-a619-f2ff68944a28


  7%|▋         | 2/27 [05:09<1:05:10, 156.41s/it]

...

 96%|█████████▋| 26/27 [1:12:10<02:43, 163.30s/it]

Generation completed
Prompt ID: 1442bda7-05ba-4dad-a0bb-a5ccad81c922


100%|██████████| 27/27 [1:14:53<00:00, 166.44s/it]

Generation completed


<Response [200]>

In [5]:
for product in products:
    timestamp_ms = int(time.time() * 1000)
    product['title'] = f"{product['title']} | PiyaGuJwells"
    product['subtitle'] = f"{product['subtitle']} | PiyaGuJwells"
    product['image_path'] = shutil.move(product['image_path'], ORIGINAL_IMAGES_DIR)
    product['GENERATED_IMAGE_LOCATION'] = shutil.move(product['GENERATED_IMAGE_LOCATION'], GENERATED_IMAGES_DIR)
    product["price"] = product["price"]*1.8
    products_db.add_product(product, timestamp_ms, product['image_path'], product['GENERATED_IMAGE_LOCATION'])

## UI

In [2]:

sys.path.insert(0, 'upload_to_shopify')  # folder containing product_editor.py and db.py

def run():
    app = QApplication.instance() or QApplication(sys.argv)
    app.setStyle('Fusion')
    win = pe.ProductEditorWindow(db_path='products.db')
    win.show()
    app.exec_()

threading.Thread(target=run, daemon=True).start()

## Upload

In [3]:
def check_errors(results):
    for k,v in results.items():
        if v is None or k == "shopify_image_url":
            continue
        for k2,v2 in v["data"].items():
            if v2["userErrors"]:
                print(v["data"])
                return 1
    return 0
while True:
    p = products_db.get_next_ready_for_upload_product()
    if p is None:
        break
    if p["price"] <500:
        print("check_price for ",p)
        break
    results = sp.upload_product(p)
    if check_errors(results):
        break
    products_db.mark_uploaded(p["id"], results['shopify_image_url'], results["product_upload_response"]["data"]["productCreate"]["product"]["id"])


['gid://shopify/Collection/311683285131', 'gid://shopify/Collection/317244014731', 'gid://shopify/Collection/317244080267', 'gid://shopify/Collection/317244178571']
['gid://shopify/Collection/311458955403', 'gid://shopify/Collection/317244080267', 'gid://shopify/Collection/317244178571', 'gid://shopify/Collection/317244276875']
['gid://shopify/Collection/311458955403', 'gid://shopify/Collection/317244014731', 'gid://shopify/Collection/317244178571']


## Regenerate

In [8]:
products_left = products_db.get_regenerate_image_products()
workflow = app.load_workflow()
comfy_prompts = app.load_prompts()
for i, product in enumerate(tqdm.tqdm(products_left)):
    generated_image_prefix = f"generated_{int(time.time() * 1000)}_{i}_"

    prompt = comfy_prompts["Alternative"].replace(
        "{set_includes}",
        ", ".join(product["set_includes"])
    )

    app.run_workflow(
        image_path=product["original_image_path"],
        prompt=prompt,
        workflow=workflow,
        output_prefix=generated_image_prefix
    )

    if os.path.exists(product["generated_image_path"]):
        os.remove(product["generated_image_path"])

    GENERATED_IMAGE_LOCATION = os.path.join(COMFY_OUTPUT_IMAGES_DIR,generated_image_prefix+"_00001_.png")
    GENERATED_IMAGE_LOCATION = shutil.move(GENERATED_IMAGE_LOCATION, GENERATED_IMAGES_DIR)
    products_db.update_product_generated_image_path(product["id"],GENERATED_IMAGE_LOCATION)
    products_db.mark_for_regeneration(product["id"],False)
app.free_resources()


  0%|          | 0/4 [00:00<?, ?it/s]

Prompt ID: 5dcc81d5-0736-4eed-a5c4-5e3727316264


 25%|██▌       | 1/4 [02:35<07:45, 155.18s/it]

Generation completed
Prompt ID: 22d35202-e646-4050-bfd9-0fc82755c876


 50%|█████     | 2/4 [05:33<05:37, 168.75s/it]

...

 75%|███████▌  | 3/4 [08:29<02:51, 171.98s/it]

Generation completed
Prompt ID: c6c96689-b68d-4971-8bd7-e27fc4cc982f


100%|██████████| 4/4 [11:43<00:00, 175.75s/it]

Generation completed


<Response [200]>

In [9]:
product_data = dict({})
product_data["product_tags"] = ["shirts", "summer"]
product_data["product_tags"].append("new-arrivals")
product_data["product_tags"]

['shirts', 'summer', 'new-arrivals']

[{'id': 18,
  'sku': 1781804981518,
  'title': 'Antique Polish Premium Quality Brass Metal Kempu Stone Exclusive Nagas Work Shiv Trishul with Shivling & Shiva Parvathi Pendant Set Combo Set PC 3090 | PiyaGuJwells',
  'subtitle': 'Exclusive Divine Combo Set with Antique Finish | PiyaGuJwells',
  'meta_title': 'Antique Polish Brass Metal Shiv Trishul & Parvathi Pendant Set PC 3090',
  'meta_description': 'Shop the exclusive PC 3090 combo set featuring antique polish brass metal with Kempu stones. Includes Shiv Trishul, Nagas work Shivling, and Shiva Parvathi pendant.',
  'alt_text': 'Antique Polish Brass Metal Shiv Trishul with Shivling and Shiva Parvathi Pendant Set',
  'variants': [],
  'product_type': 'Jewelry Sets',
  'product_tags': ['festive', 'gifting'],
  'set_includes': ['Shiv Trishul',
   'Shivling with Nagas Work',
   'Shiva & Parvathi Pendant'],
  'price': 0.0,
  'original_image_path': 'c:\\Users\\FIRST\\Desktop\\piyagushopify\\llmautomation\\original_images\\WhatsApp Image 2